# Segmentation Pipeline: Assets -> Labels -> ML -> MLOps

Ten notebook jest ułożony jako pełny workflow:

1. Inicjalizacja środowiska i ścieżek
2. Demo: generowanie screenów i labelowanie
3. Uczenie maszynowe (segmentation U-Net)
4. Integracja z MLOps (MLflow)
5. Statystyki i wizualizacja działania modelu

Uruchamiaj komórki po kolei od góry.

In [1]:
# 1) Inicjalizacja środowiska
import os
from pathlib import Path
import json
import time
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from PIL import Image
import torch
import cv2
import mlflow

PROJECT_ROOT = Path.cwd().resolve()

# Ensure the project `src` package is importable when running the notebook
import sys
if not (PROJECT_ROOT / 'src').exists():
    p = PROJECT_ROOT.resolve()
    found = False
    while True:
        if (p / 'src').exists():
            PROJECT_ROOT = p
            sys.path.insert(0, str(PROJECT_ROOT))
            found = True
            break
        if p == p.parent:
            break
        p = p.parent
    if not found:
        sys.path.insert(0, str(PROJECT_ROOT))
else:
    sys.path.insert(0, str(PROJECT_ROOT))

# Paths derived from resolved repository root
TRAIN_DIR = PROJECT_ROOT / 'data' / 'segmentation' / 'train'
TEST_DIR = PROJECT_ROOT / 'data' / 'segmentation' / 'test'
REAL_SCREENSHOTS_DIR = PROJECT_ROOT / 'src' / 'dataset' / 'Real_screenshots'
OUT_DIR = PROJECT_ROOT / 'data' / 'real_screenshot_predictions'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_DB = (PROJECT_ROOT / 'mlruns' / 'mlflow.db').resolve()
MLFLOW_TRACKING_URI = f'sqlite:///{MLFLOW_DB}'

# local package imports
from src.dataset.pacman_map_dataset import PacmanMapDatasetGenerator, CLASS_TO_ID
from src.environment.game_logic import GameState
from src.models.segmentation_detector import (
    SegmentationDetector, TrainConfig, SegmentationDataset,
    extract_instances, remap_instances_to_image,
    detect_playfield_bbox, parse_hud_numbers, ID_TO_CLASS,
)
from src.utils.mlflow_logger import MLflowLogger

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Train dir:', TRAIN_DIR)
print('Test dir:', TEST_DIR)
print('Real screenshots dir:', REAL_SCREENSHOTS_DIR)
print('MLflow tracking URI:', MLFLOW_TRACKING_URI)


Device: cuda
Train dir: /home/mwrona/PAC-MAN-AI/data/segmentation/train
Test dir: /home/mwrona/PAC-MAN-AI/data/segmentation/test
Real screenshots dir: /home/mwrona/PAC-MAN-AI/src/dataset/Real_screenshots
MLflow tracking URI: sqlite:////home/mwrona/PAC-MAN-AI/mlruns/mlflow.db


In [2]:
# 4) Połączenie z MLOps (MLflow) - bootstrap
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print('MLflow DB exists:', MLFLOW_DB.exists())

with MLflowLogger(experiment_name='segmentation_training', run_name='segmentation_bootstrap') as logger:
    logger.log_params({
        'source_notebook': '06_07_08',
        'device': DEVICE,
        'train_dir': str(TRAIN_DIR),
        'test_dir': str(TEST_DIR),
    })
    logger.log_metric('bootstrap_ok', 1.0)

print('MLOps bootstrap run zapisany.')

MLflow DB exists: True
MLOps bootstrap run zapisany.


In [ ]:
# 2) Demo: generowanie screenów i podgląd labelowania
import ipywidgets as widgets
from IPython.display import display
gen = PacmanMapDatasetGenerator()

seed_slider = widgets.IntSlider(description='Seed', min=0, max=2**31-1, value=0)
steps_slider = widgets.IntSlider(description='Rollout steps', min=0, max=200, value=50)
btn = widgets.Button(description='Render sample')
out = widgets.Output()

def render_sample(seed, steps):
    state = GameState(seed=int(seed))
    for _ in range(int(steps)):
        if state.is_terminal():
            break
        state.step(int(np.random.default_rng().integers(0, 4)))
    frame, annotation, mask = gen.render_state(state)
    return frame, mask, annotation

def on_click(_):
    with out:
        out.clear_output(wait=True)
        frame, mask, ann = render_sample(seed_slider.value, steps_slider.value)
        display(frame)
        display(mask)
        print('HUD:', ann.get('hud'))
        print('Instances:', len(ann.get('instances', [])))

btn.on_click(on_click)
display(widgets.VBox([seed_slider, steps_slider, btn, out]))

In [3]:
# Visualization helpers (for labeling + inference preview)
def _group_for_label(label: str) -> str:
    if label in {'blinky', 'pinky', 'inky', 'clyde', 'frightened_ghost', 'ghost_eyes'}:
        return 'ghosts'
    if label == 'fruit':
        return 'fruit'
    if label == 'pacman':
        return 'pacman'
    if label in {'pellet', 'power_pellet'}:
        return 'collectibles'
    return 'other'

GROUP_COLORS = {
    'pacman': '#ffd400',
    'ghosts': '#5ac8fa',
    'fruit': '#ff7a59',
    'collectibles': '#b8f397',
    'other': '#ffffff',
}

def show_annotated_image(image: np.ndarray, instances: list, playfield_bbox=None, figsize=(14, 12), show=True):
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    ax.imshow(image)
    if playfield_bbox is not None:
        x0, y0, bw, bh = playfield_bbox
        ax.add_patch(Rectangle((x0, y0), bw, bh, fill=False, edgecolor='#00e5ff', linewidth=3))
    for obj in instances:
        x, y, w, h = obj['bbox']
        label = obj['label']
        group = _group_for_label(label)
        color = GROUP_COLORS.get(group, '#ffffff')
        ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor=color, linewidth=2))
        ax.text(
            x,
            max(0, y - 6),
            label,
            color=color,
            fontsize=9,
            weight='bold',
            bbox=dict(facecolor='black', alpha=0.6, pad=1, edgecolor='none'),
        )
    ax.axis('off')
    if show:
        plt.show()
    return fig, ax

In [4]:
# Generate datasets (if missing)
if not (TRAIN_DIR / 'images').exists():
    print('Generating training dataset... (2000 samples)')
    gen = PacmanMapDatasetGenerator()
    gen.generate_dataset(TRAIN_DIR, sample_count=2000, seed=0, max_random_steps=200)
else:
    print('Train dataset already exists, skipping generation')

if not (TEST_DIR / 'images').exists():
    print('Generating test dataset... (500 samples)')
    gen = PacmanMapDatasetGenerator()
    gen.generate_dataset(TEST_DIR, sample_count=500, seed=1, max_random_steps=200)
else:
    print('Test dataset already exists, skipping generation')

Train dataset already exists, skipping generation
Test dataset already exists, skipping generation


In [ ]:
# 3) Uczenie maszynowe (segmentation U-Net) + logowanie do MLOps
model_out = MODELS_DIR / 'segmentation_unet_combined.pt'
TRAIN_EPOCHS = 300
TRAIN_BATCH_SIZE = 8
TRAIN_LR = 1e-3
TRAIN_VAL_SPLIT = 0.1
TRAIN_SEED = 42

with MLflowLogger(experiment_name='segmentation_training', run_name='segmentation_unet_train') as logger:
    logger.log_params({
        'model_out': str(model_out),
        'dataset_dir': str(TRAIN_DIR),
        'epochs': TRAIN_EPOCHS,
        'batch_size': TRAIN_BATCH_SIZE,
        'lr': TRAIN_LR,
        'val_split': TRAIN_VAL_SPLIT,
        'seed': TRAIN_SEED,
        'device': DEVICE,
    })

    if model_out.exists():
        print('Model already exists at', model_out, '- skipping training')
        logger.log_metric('training_skipped_existing_model', 1.0)
    else:
        cfg = TrainConfig(
            dataset_dir=TRAIN_DIR,
            output_path=model_out,
            epochs=TRAIN_EPOCHS,
            batch_size=TRAIN_BATCH_SIZE,
            lr=TRAIN_LR,
            val_split=TRAIN_VAL_SPLIT,
            seed=TRAIN_SEED,
            device=DEVICE,
        )
        detector = SegmentationDetector(device=DEVICE)
        print('Starting training — this can take a long time. Reduce epochs for quick tests.')
        t0 = time.time()
        metrics = detector.train(cfg)
        t1 = time.time()

        train_seconds = t1 - t0
        logger.log_metrics({
            'train_loss': float(metrics.get('train_loss', np.nan)),
            'val_loss': float(metrics.get('val_loss', np.nan)),
            'best_val_loss': float(metrics.get('best_val_loss', np.nan)),
            'train_time_sec': float(train_seconds),
        })

        print('Done. Time (s):', train_seconds)
        print('Metrics:', metrics)


Starting training — this can take a long time. Reduce epochs for quick tests.


In [ ]:
# 5) Statystyki jakości modelu: mIoU + per-class IoU
model_path = model_out
if not model_path.exists():
    print('Model not found at', model_path, '- run training cell first or point to existing checkpoint')
else:
    detector = SegmentationDetector.load(model_path, device=DEVICE)
    test_ds = SegmentationDataset(TEST_DIR)
    print('Running mIoU computation on test set (this may take a few minutes)')

    def compute_miou(detector, dataset, id_to_class):
        class_ids = sorted(id_to_class.keys())
        intersection = {cid: 0 for cid in class_ids}
        union = {cid: 0 for cid in class_ids}
        for idx in range(len(dataset)):
            img_t, mask_t = dataset[idx]
            img_np = (img_t.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            pred = detector.predict_mask(img_np)
            gt = mask_t.numpy().astype(np.int32)
            for cid in class_ids:
                if cid == 0:
                    continue
                pred_c = (pred == cid).astype(np.uint8)
                gt_c = (gt == cid).astype(np.uint8)
                inter = int((pred_c & gt_c).sum())
                uni = int((pred_c | gt_c).sum())
                intersection[cid] += inter
                union[cid] += uni

        ious = {}
        for cid in class_ids:
            if cid == 0:
                continue
            if union[cid] > 0:
                ious[id_to_class[cid]] = float(intersection[cid]) / float(union[cid])
            else:
                ious[id_to_class[cid]] = None
        vals = [v for v in ious.values() if v is not None]
        miou = float(np.mean(vals)) if vals else None
        return ious, miou

    per_class_iou, miou = compute_miou(detector, test_ds, ID_TO_CLASS)
    report = {'per_class_iou': per_class_iou, 'miou': miou}
    (PROJECT_ROOT / 'data' / 'segmentation' / 'pipeline_metrics.json').write_text(
        json.dumps(report, indent=2),
        encoding='utf-8',
    )

    print('mIoU:', miou)
    print('Per-class IoU:')
    for k, v in per_class_iou.items():
        print(' ', k, ':', v)

    with MLflowLogger(experiment_name='segmentation_training', run_name='segmentation_eval') as logger:
        logger.log_params({'model_path': str(model_path), 'dataset': str(TEST_DIR)})
        metric_payload = {'miou': float(miou) if miou is not None else np.nan}
        for label, value in per_class_iou.items():
            metric_payload[f'iou_{label}'] = float(value) if value is not None else np.nan
        logger.log_metrics(metric_payload)

    # Bar chart per-class IoU
    plot_items = [(k, v) for k, v in per_class_iou.items() if v is not None]
    if plot_items:
        labels, values = zip(*plot_items)
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.bar(labels, values, color='steelblue')
        ax.set_ylim(0, 1)
        ax.set_ylabel('IoU')
        ax.set_title(f'Per-class IoU (mIoU={miou:.3f})')
        ax.tick_params(axis='x', rotation=45)
        plt.tight_layout()
        plt.show()

Running mIoU computation on test set (this may take a few minutes)
mIoU: 0.9994178650430283
Per-class IoU:
  wall : 1.0
  pellet : 0.99999682238895
  power_pellet : 0.9998108612903989
  ghost_door : 1.0
  ghost_house : 1.0
  pacman : 0.9971820759283591
  blinky : 0.9996890879589596
  pinky : 0.9995564028994262
  inky : 0.9999339660019244
  clyde : 0.9980094339622642
  frightened_ghost : None
  ghost_eyes : None
  fruit : None


In [ ]:
# Inference on real screenshots (annotated)
model_path = model_out
if not model_path.exists():
    print('Model not found at', model_path, '- train first or point to an existing checkpoint')
else:
    detector = SegmentationDetector.load(model_path, device=DEVICE)
    for img_path in sorted(REAL_SCREENSHOTS_DIR.glob('*.png')):
        stem = img_path.stem
        outdir = OUT_DIR / stem
        outdir.mkdir(parents=True, exist_ok=True)
        image = np.asarray(Image.open(img_path).convert('RGB'), dtype=np.uint8)
        playfield_bbox = detect_playfield_bbox(image)
        x0, y0, bw, bh = playfield_bbox
        crop = image[y0:y0+bh, x0:x0+bw]
        resized = cv2.resize(crop, (224, 248), interpolation=cv2.INTER_AREA)
        pred_mask = detector.predict_mask(resized)
        instances_local = extract_instances(pred_mask, ID_TO_CLASS, min_area=10)
        instances = remap_instances_to_image(instances_local, playfield_bbox, pred_mask.shape)
        hud = {'lives_icons': [], }
        hud.update(parse_hud_numbers(image))
        # save masks and json
        mask_playfield_path = outdir / 'pred_mask_playfield.png'
        Image.fromarray(pred_mask, mode='L').save(mask_playfield_path)
        full_mask = np.zeros(image.shape[:2], dtype=np.uint8)
        restored = cv2.resize(pred_mask, (bw, bh), interpolation=cv2.INTER_NEAREST)
        full_mask[y0:y0+bh, x0:x0+bw] = restored
        full_mask_path = outdir / 'pred_mask_full.png'
        Image.fromarray(full_mask, mode='L').save(full_mask_path)
        result = {
            'image': str(img_path),
            'playfield_bbox': [int(v) for v in playfield_bbox],
            'mask_playfield': str(mask_playfield_path),
            'mask_full': str(full_mask_path),
            'instances': instances,
            'hud': hud,
        }
        (outdir / 'prediction_arcade.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
        show_annotated_image(image, instances, playfield_bbox=playfield_bbox)
        print('Saved prediction for', stem, '->', outdir)

In [ ]:
# Single real screenshot inference + labeled view + save overlay
model_path = model_out
img_path = REAL_SCREENSHOTS_DIR / 'Screenshot_20260614_131655.png'

if not model_path.exists():
    print('Model not found:', model_path)
else:
    detector = SegmentationDetector.load(model_path, device=DEVICE)

    stem = img_path.stem
    outdir = OUT_DIR / stem
    outdir.mkdir(parents=True, exist_ok=True)

    image = np.asarray(Image.open(img_path).convert('RGB'), dtype=np.uint8)
    playfield_bbox = detect_playfield_bbox(image)
    x0, y0, bw, bh = playfield_bbox

    crop = image[y0:y0+bh, x0:x0+bw]
    resized = cv2.resize(crop, (224, 248), interpolation=cv2.INTER_AREA)
    pred_mask = detector.predict_mask(resized)

    instances_local = extract_instances(pred_mask, ID_TO_CLASS, min_area=10)
    instances = remap_instances_to_image(instances_local, playfield_bbox, pred_mask.shape)

    hud = {'lives_icons': []}
    hud.update(parse_hud_numbers(image))

    # save masks/json
    mask_playfield_path = outdir / 'pred_mask_playfield.png'
    Image.fromarray(pred_mask, mode='L').save(mask_playfield_path)

    full_mask = np.zeros(image.shape[:2], dtype=np.uint8)
    restored = cv2.resize(pred_mask, (bw, bh), interpolation=cv2.INTER_NEAREST)
    full_mask[y0:y0+bh, x0:x0+bw] = restored
    full_mask_path = outdir / 'pred_mask_full.png'
    Image.fromarray(full_mask, mode='L').save(full_mask_path)

    result = {
        'image': str(img_path),
        'playfield_bbox': [int(v) for v in playfield_bbox],
        'mask_playfield': str(mask_playfield_path),
        'mask_full': str(full_mask_path),
        'instances': instances,
        'hud': hud,
    }
    (outdir / 'prediction_arcade.json').write_text(json.dumps(result, indent=2), encoding='utf-8')

    # labeled screenshot preview + save
    fig, _ = show_annotated_image(image, instances, playfield_bbox=playfield_bbox, show=True)
    fig.savefig(outdir / 'annotated_overlay.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    with MLflowLogger(experiment_name='segmentation_training', run_name='segmentation_infer_single') as logger:
        logger.log_params({'model_path': str(model_path), 'image': str(img_path)})
        logger.log_metrics({'n_instances': float(len(instances))})

    print('Saved:', outdir)

FileNotFoundError: [Errno 2] No such file or directory: '/home/mwrona/PAC-MAN-AI/notebooks/src/dataset/Real_screenshots/Screenshot_20260614_131655.png'

In [ ]:
# MLOps stats dashboard: ostatnie runy segmentacji
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
exp = client.get_experiment_by_name('segmentation_training')

if exp is None:
    print("Experiment 'segmentation_training' not found")
else:
    runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        order_by=['attributes.start_time DESC'],
        max_results=30,
    )

    rows = []
    for r in runs:
        rows.append({
            'run_name': r.data.tags.get('mlflow.runName', r.info.run_id[:8]),
            'status': r.info.status,
            'start_time': pd.to_datetime(r.info.start_time, unit='ms'),
            'train_loss': r.data.metrics.get('train_loss', np.nan),
            'val_loss': r.data.metrics.get('val_loss', np.nan),
            'best_val_loss': r.data.metrics.get('best_val_loss', np.nan),
            'miou': r.data.metrics.get('miou', np.nan),
            'n_instances': r.data.metrics.get('n_instances', np.nan),
        })

    df_runs = pd.DataFrame(rows)
    display(df_runs.head(20))

    # Visual summary for recent runs
    if not df_runs.empty:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        train_df = df_runs.dropna(subset=['best_val_loss']).copy().sort_values('start_time')
        if not train_df.empty:
            axes[0].plot(train_df['start_time'], train_df['best_val_loss'], marker='o', color='tab:blue')
            axes[0].set_title('Best val loss over time')
            axes[0].set_ylabel('best_val_loss')
            axes[0].tick_params(axis='x', rotation=30)
        else:
            axes[0].set_title('Best val loss over time')
            axes[0].text(0.5, 0.5, 'No train runs', ha='center', va='center')
            axes[0].set_axis_off()

        eval_df = df_runs.dropna(subset=['miou']).copy().sort_values('start_time')
        if not eval_df.empty:
            axes[1].plot(eval_df['start_time'], eval_df['miou'], marker='o', color='tab:green')
            axes[1].set_title('mIoU over time')
            axes[1].set_ylabel('mIoU')
            axes[1].set_ylim(0, 1)
            axes[1].tick_params(axis='x', rotation=30)
        else:
            axes[1].set_title('mIoU over time')
            axes[1].text(0.5, 0.5, 'No eval runs', ha='center', va='center')
            axes[1].set_axis_off()

        plt.tight_layout()
        plt.show()

## Run Order (quick checklist)

1. Inicjalizacja środowiska
2. MLOps bootstrap (sprawdzenie MLflow)
3. Demo generowania i labelowania (widget + dataset generation)
4. Uczenie modelu segmentacji
5. Ewaluacja (mIoU + IoU per klasa)
6. Inference na real screenshotach + zapis overlay
7. Dashboard statystyk z MLOps

Dzięki temu masz pełny flow od assetów i labeli aż do monitoringu metryk.